# Dự án: Phân tích ứng dụng Google Play Store
## Đề tài: Dự báo điểm Rating ứng dụng dựa trên phân tích Cảm xúc người dùng (Sentiment Analysis)
### Notebook 06: Machine Learning (06_machine_learning.ipynb)

# I. Giới thiệu

**Mục tiêu của Notebook:** Huấn luyện, đánh giá và so sánh nhiều mô hình Machine Learning để dự báo điểm **`Rating`** của ứng dụng trên Google Play Store, dựa trên bộ đặc trưng (`train_features` / `test_features`) đã được xây dựng ở Notebook 05. Từ đó lựa chọn ra mô hình tốt nhất để triển khai ở Notebook 07.

**Vai trò của Machine Learning trong dự án:** đây là bước hiện thực hóa mục tiêu cuối cùng của cả pipeline — biến các đặc trưng đã xây dựng (Notebook 05) thành một mô hình dự báo cụ thể, có thể đo lường được bằng số liệu (MAE, RMSE, R²) và có thể tái sử dụng (lưu thành file `.pkl`) ở bước triển khai.

**Mối liên hệ giữa Notebook 05 và Notebook 07:**

```
Notebook 05: Feature Engineering  (tạo bảng train_features / test_features trên PostgreSQL)
        ↓
Notebook 06: Machine Learning     (đang ở đây — huấn luyện, đánh giá, chọn mô hình, lưu model.pkl)
        ↓
Notebook 07: Prediction Demo      (đọc model.pkl, xây dựng chương trình dự báo + Dashboard)
```

**Lưu ý về đề bài gốc:** khung sườn ban đầu của Notebook 06 được viết cho bài toán *dự báo doanh số* (đọc bảng `sales_features`, chia Train/Test theo thời gian). Dự án của nhóm là bài toán *dự báo Rating ứng dụng* từ dữ liệu Google Play Store (không có yếu tố thời gian dạng chuỗi doanh số), nên notebook này được điều chỉnh cho đúng với dữ liệu thực tế của nhóm:
- Đọc từ bảng **`train_features` / `test_features`** (không phải `sales_features`) — do Notebook 05 tạo ra.
- **Không chia lại Train/Test theo thời gian** — vì Train/Test đã được chia ngẫu nhiên theo tỷ lệ 80/20 từ Notebook 03, và toàn bộ pipeline (04 → 05 → 06) luôn giữ nguyên đúng cách chia này để đảm bảo tính nhất quán, tránh rò rỉ dữ liệu (data leakage) giữa các bước.

# II. Đọc dữ liệu

Theo đúng nguyên tắc đã áp dụng từ Notebook 04: dữ liệu **bắt buộc đọc từ PostgreSQL**, không đọc lại CSV. Notebook 05 đã tạo sẵn 2 bảng `train_features` và `test_features` — đây chính là bộ dữ liệu đầu vào cho toàn bộ Notebook 06.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')

# --- Cấu hình kết nối PostgreSQL (PHẢI khớp với Notebook 02/03/04/05) ---
DB_USER = "postgres"
DB_PASS = "hoai123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Du_an_1_valuevoice"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# --- Đọc dữ liệu đặc trưng TRỰC TIẾP từ PostgreSQL (bảng do Notebook 05 tạo) ---
# Nếu bước này báo lỗi, cần kiểm tra: (a) PostgreSQL server đã chạy chưa,
# (b) Notebook 05 đã chạy xong cell ghi bảng train_features/test_features chưa.
df_train = pd.read_sql("SELECT * FROM train_features;", engine)
df_test = pd.read_sql("SELECT * FROM test_features;", engine)

print("Đã đọc dữ liệu thành công từ PostgreSQL (bảng train_features / test_features).")
print(f"\nTập Train: {df_train.shape} | Tập Test: {df_test.shape}")
df_train.head(3)

**Nhận xét:**
- Bộ dữ liệu đọc được ở đây là bộ đặc trưng đã hoàn chỉnh từ Notebook 05 (đã log-transform, điền khuyết, mã hóa Category/Type/Content Rating), **không phải** dữ liệu thô như ở Notebook 04.
- Số dòng của `train_features`/`test_features` phải khớp đúng với `train_clean`/`test_clean` ở Notebook 03/04 (không có dòng nào bị mất trong quá trình Feature Engineering) — nếu lệch số dòng, cần kiểm tra lại Notebook 05.

In [ ]:
print("--- KIỂU DỮ LIỆU TỪNG CỘT (Tập Train) ---")
df_train.info()

print("\n--- SỐ LƯỢNG GIÁ TRỊ KHUYẾT THIẾU THEO CỘT ---")
missing = df_train.isnull().sum()
missing = missing[missing > 0]
print(missing if len(missing) > 0 else "Không còn cột nào bị khuyết thiếu — đúng như đã đảm bảo ở Notebook 05, mục X.")

**Nhận xét:**
- Nếu kết quả in ra không còn cột nào khuyết thiếu, đây là bằng chứng xác nhận bước "chốt chặn missing" ở Notebook 05 (mục X) đã hoạt động đúng — dữ liệu sẵn sàng để đưa vào mô hình mà không cần xử lý thêm.
- Các cột `App`, `Category`, `Primary_Genre` vẫn còn là kiểu chuỗi (object) — đây là các cột **giữ lại để tham chiếu/kiểm tra**, không phải Feature đưa vào mô hình (xem mục III).

## Xác định Feature và Target

- **Target (biến mục tiêu):** `Rating` — đây là điểm số ứng dụng mà mô hình cần dự báo.
- **Feature (biến đầu vào):** toàn bộ các cột số đã được xử lý ở Notebook 05 (`Reviews_log`, `Installs_log`, `Size`, `Price`, `has_review`, `Sentiment_Polarity`, `Sentiment_Subjectivity`, `Days_Since_Update`, `Category_TargetEnc`, cùng các cột One-Hot `Type_*`/`CR_*`).
- **Không đưa vào mô hình:** `App` (định danh, không mang thông tin dự báo), `Category` và `Primary_Genre` (dạng chuỗi thô, chưa mã hóa — thông tin của `Category` đã được biểu diễn an toàn hơn qua `Category_TargetEnc`).

# III. Chuẩn bị dữ liệu

## Lựa chọn Feature đầu vào & loại bỏ cột không sử dụng

Cột Feature được xác định **tự động theo quy tắc loại trừ**, thay vì gõ tay từng tên cột — cách này giúp notebook vẫn chạy đúng nếu Notebook 05 có thay đổi nhỏ về số lượng cột One-Hot (`Type_*`/`CR_*`) mà không cần sửa lại code ở đây.

In [ ]:
COT_LOAI_TRU = ['App', 'Category', 'Primary_Genre', 'Rating']

feature_cols = [c for c in df_train.columns if c not in COT_LOAI_TRU]
target_col = 'Rating'

print(f"Số lượng Feature sử dụng cho mô hình: {len(feature_cols)}")
print("Danh sách Feature:")
for c in feature_cols:
    print(f"  - {c}")

X_train = df_train[feature_cols].copy()
y_train = df_train[target_col].copy()
X_test = df_test[feature_cols].copy()
y_test = df_test[target_col].copy()

print(f"\nX_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

**Nhận xét:**
- Việc chọn Feature bằng quy tắc loại trừ (`COT_LOAI_TRU`) thay vì liệt kê thủ công giúp code bền vững hơn với các thay đổi nhỏ ở Notebook 05 (ví dụ số cột One-Hot `CR_*` phụ thuộc vào số giá trị duy nhất của `Content Rating` trong Train).
- `X_train`/`X_test` chỉ chứa các cột số đã qua xử lý (không còn chuỗi thô), sẵn sàng đưa trực tiếp vào các mô hình `scikit-learn`/`XGBoost` mà không cần encode thêm.

## Về việc chia Train/Test

Đề bài mẫu yêu cầu chia Train/Test theo thời gian (Time-based Split) — cách này phù hợp với dữ liệu dạng chuỗi thời gian (ví dụ doanh số bán hàng theo ngày, nơi Test phải luôn là giai đoạn **sau** Train). Dữ liệu của nhóm là một **bức ảnh chụp (snapshot)** các ứng dụng trên Google Play Store tại một thời điểm, không phải dữ liệu quan sát lặp lại theo thời gian cho từng ứng dụng, nên chia theo thời gian không áp dụng được ở đây.

Thay vào đó, nhóm giữ nguyên cách chia Train/Test đã thực hiện nhất quán từ Notebook 03 (chia ngẫu nhiên theo tỷ lệ 80/20, `random_state` cố định để tái lập được) và tuân thủ xuyên suốt các Notebook 04 → 05 → 06. Đây là cách chia phù hợp và đủ chặt chẽ cho một bộ dữ liệu dạng snapshot như thế này, miễn là Test không hề được "nhìn thấy" trong bất kỳ bước học tham số nào trước đó — điều mà nhóm đã đảm bảo xuyên suốt (xem Notebook 05, mục I và VI.2).

# IV. Giới thiệu các mô hình

Nhóm huấn luyện và so sánh 3 mô hình đại diện cho 3 mức độ phức tạp khác nhau: một mô hình tuyến tính đơn giản (baseline) và hai mô hình dạng cây/ensemble mạnh hơn.

## 1. Linear Regression

**Giới thiệu:** mô hình hồi quy tuyến tính cơ bản, giả định `Rating` là một tổ hợp tuyến tính (cộng có trọng số) của các Feature đầu vào.

**Ưu điểm:** đơn giản, huấn luyện rất nhanh, dễ diễn giải (mỗi hệ số cho biết mức ảnh hưởng của 1 Feature), là mốc so sánh (baseline) tốt để biết các mô hình phức tạp hơn có thực sự "đáng" phức tạp hơn hay không.

**Trường hợp sử dụng:** phù hợp khi quan hệ giữa Feature và Target gần như tuyến tính. Theo phát hiện ở Notebook 04 (mục V), `Rating` hầu như không tương quan tuyến tính mạnh với các biến số gốc — nên mô hình này được kỳ vọng đóng vai trò baseline hơn là lựa chọn cuối cùng.

## 2. Random Forest Regressor

**Giới thiệu:** một ensemble gồm nhiều cây quyết định (decision tree), mỗi cây học trên một mẫu ngẫu nhiên (bootstrap) của dữ liệu và một tập con ngẫu nhiên của Feature; kết quả dự báo cuối cùng là trung bình dự báo của toàn bộ các cây.

**Ưu điểm:** học được quan hệ phi tuyến và tương tác giữa các Feature (đúng như insight ở Notebook 04, mục VII), ít nhạy với outlier, ít cần tinh chỉnh siêu tham số để có kết quả khá tốt ngay từ đầu, đồng thời cung cấp sẵn Feature Importance.

**Trường hợp sử dụng:** phù hợp với dữ liệu có nhiều biến phân loại/tương tác phức tạp như bộ dữ liệu này — nơi các biến số gốc tương quan tuyến tính yếu với Rating nhưng tương tác giữa Category/Type/Content Rating lại có ảnh hưởng rõ hơn.

## 3. XGBoost Regressor

**Giới thiệu:** một ensemble dạng Gradient Boosting — khác với Random Forest (các cây học độc lập), các cây trong XGBoost được xây dựng **tuần tự**, mỗi cây mới tập trung sửa lỗi (residual) mà các cây trước đó chưa dự báo đúng.

**Ưu điểm:** thường cho độ chính xác cao hơn Random Forest trên nhiều bài toán dạng bảng (tabular data) nhờ cơ chế boosting, có nhiều tham số điều chỉnh (learning rate, độ sâu cây, regularization) giúp kiểm soát tốt hiện tượng overfitting nếu tinh chỉnh đúng.

**Trường hợp sử dụng:** phù hợp khi cần độ chính xác cao và có thời gian/tài nguyên để tinh chỉnh siêu tham số; thường là lựa chọn phổ biến trong các cuộc thi và ứng dụng thực tế với dữ liệu dạng bảng.

# V. Huấn luyện mô hình

Với mỗi mô hình, quy trình gồm 3 bước: **Huấn luyện (Train)** trên `X_train`/`y_train` → **Dự đoán (Predict)** trên `X_test` → **Đánh giá (Evaluate)** bằng MAE/RMSE/R². Kết quả từng mô hình được lưu vào dictionary `ket_qua_mo_hinh` để tổng hợp so sánh ở mục VI/IX.

In [ ]:
def danh_gia_mo_hinh(y_thuc, y_du_bao):
    '''Tính 3 chỉ số đánh giá cho một tập dự báo.'''
    mae = mean_absolute_error(y_thuc, y_du_bao)
    rmse = np.sqrt(mean_squared_error(y_thuc, y_du_bao))
    r2 = r2_score(y_thuc, y_du_bao)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Nơi lưu kết quả (metrics, thời gian huấn luyện, model đã fit, dự báo) của TỪNG mô hình
ket_qua_mo_hinh = {}

## 1. Linear Regression

In [ ]:
t0 = time.time()
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
thoi_gian_train_lr = time.time() - t0

y_pred_lr = lr_model.predict(X_test)
metrics_lr = danh_gia_mo_hinh(y_test, y_pred_lr)

ket_qua_mo_hinh['Linear Regression'] = {
    'model': lr_model,
    'y_pred': y_pred_lr,
    'train_time': thoi_gian_train_lr,
    **metrics_lr,
}

print(f"Thời gian huấn luyện: {thoi_gian_train_lr:.3f} giây")
print(f"MAE  = {metrics_lr['MAE']:.4f}")
print(f"RMSE = {metrics_lr['RMSE']:.4f}")
print(f"R2   = {metrics_lr['R2']:.4f}")

**Nhận xét:** đây là kết quả baseline. Nếu R² của Linear Regression khá thấp (gần 0 hoặc âm), điều này khớp với phát hiện ở Notebook 04 (mục V) rằng `Rating` không có quan hệ tuyến tính mạnh với các biến số — nghĩa là một mô hình học được quan hệ phi tuyến (Random Forest, XGBoost) có nhiều khả năng vượt trội hơn.

## 2. Random Forest Regressor

In [ ]:
t0 = time.time()
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
thoi_gian_train_rf = time.time() - t0

y_pred_rf = rf_model.predict(X_test)
metrics_rf = danh_gia_mo_hinh(y_test, y_pred_rf)

ket_qua_mo_hinh['Random Forest'] = {
    'model': rf_model,
    'y_pred': y_pred_rf,
    'train_time': thoi_gian_train_rf,
    **metrics_rf,
}

print(f"Thời gian huấn luyện: {thoi_gian_train_rf:.3f} giây")
print(f"MAE  = {metrics_rf['MAE']:.4f}")
print(f"RMSE = {metrics_rf['RMSE']:.4f}")
print(f"R2   = {metrics_rf['R2']:.4f}")

In [ ]:
# Feature Importance của Random Forest
fi_rf = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 7))
sns.barplot(x=fi_rf.values, y=fi_rf.index, hue=fi_rf.index, legend=False, palette='crest')
plt.title("Feature Importance - Random Forest", fontweight='bold')
plt.xlabel("Mức độ quan trọng")
plt.tight_layout()
plt.show()

fi_rf.to_frame('Importance')

**Nhận xét:** so sánh MAE/RMSE/R² của Random Forest với Linear Regression ở trên để xác nhận mô hình dạng cây có thực sự khai thác tốt hơn các tương tác phi tuyến (Category × Type, Content Rating × Type...) đã phát hiện ở Notebook 04 (mục VII) hay không. Thứ hạng Feature Importance ở đây cũng nên được đối chiếu với kết quả đánh giá Feature ở Notebook 05 (mục VIII) — nếu 2 lần đánh giá tương đối nhất quán, đây là dấu hiệu tốt cho thấy các Feature quan trọng thực sự ổn định, không phải ngẫu nhiên do cách chia Train/Test.

## 3. XGBoost Regressor

In [ ]:
t0 = time.time()
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
thoi_gian_train_xgb = time.time() - t0

y_pred_xgb = xgb_model.predict(X_test)
metrics_xgb = danh_gia_mo_hinh(y_test, y_pred_xgb)

ket_qua_mo_hinh['XGBoost'] = {
    'model': xgb_model,
    'y_pred': y_pred_xgb,
    'train_time': thoi_gian_train_xgb,
    **metrics_xgb,
}

print(f"Thời gian huấn luyện: {thoi_gian_train_xgb:.3f} giây")
print(f"MAE  = {metrics_xgb['MAE']:.4f}")
print(f"RMSE = {metrics_xgb['RMSE']:.4f}")
print(f"R2   = {metrics_xgb['R2']:.4f}")

**Nhận xét:** đối chiếu 3 chỉ số của XGBoost với Random Forest ngay phía trên — vì cả hai đều là ensemble dạng cây, khác biệt hiệu năng (nếu có) chủ yếu đến từ cơ chế Boosting tuần tự (sửa lỗi từng bước) so với Bagging song song của Random Forest. Nếu XGBoost không vượt trội rõ ràng, điều đó không phải là bất thường — với bộ dữ liệu tương đối nhỏ (~7,700 dòng Train) và tín hiệu dự báo yếu (R² thấp ở cả 3 mô hình), khoảng cách giữa các thuật toán ensemble thường không quá lớn.

# VI. Đánh giá mô hình

Tổng hợp lại 3 chỉ số đánh giá đã tính ở mục V cho cả 3 mô hình vào một bảng duy nhất để dễ so sánh trực tiếp.

- **MAE (Mean Absolute Error):** sai số tuyệt đối trung bình, cùng đơn vị với `Rating` (thang 0–5) — dễ diễn giải trực quan ("mô hình sai lệch trung bình khoảng bao nhiêu điểm Rating").
- **RMSE (Root Mean Squared Error):** tương tự MAE nhưng phạt nặng hơn các sai số lớn (do bình phương trước khi lấy trung bình rồi khai căn) — nhạy với outlier hơn MAE.
- **R² (R-squared):** tỷ lệ phương sai của `Rating` được mô hình giải thích được, dao động từ 0 đến 1 (có thể âm nếu mô hình dự báo kém hơn cả việc luôn đoán bằng giá trị trung bình).

In [ ]:
bang_danh_gia = pd.DataFrame({
    ten: {'MAE': tt['MAE'], 'RMSE': tt['RMSE'], 'R2': tt['R2'], 'Thời gian train (s)': tt['train_time']}
    for ten, tt in ket_qua_mo_hinh.items()
}).T.sort_values('RMSE')

bang_danh_gia

In [ ]:
mo_hinh_tot_nhat_tam = bang_danh_gia['RMSE'].idxmin()
mo_hinh_kem_nhat_tam = bang_danh_gia['RMSE'].idxmax()
chenh_lech_rmse = bang_danh_gia.loc[mo_hinh_kem_nhat_tam, 'RMSE'] - bang_danh_gia.loc[mo_hinh_tot_nhat_tam, 'RMSE']

print(f"Mô hình có RMSE thấp nhất (tạm dẫn đầu theo chỉ số này): {mo_hinh_tot_nhat_tam}")
print(f"Mô hình có RMSE cao nhất: {mo_hinh_kem_nhat_tam}")
print(f"Chênh lệch RMSE giữa 2 mô hình: {chenh_lech_rmse:.4f}")
print(f"\nR2 của {mo_hinh_tot_nhat_tam}: {bang_danh_gia.loc[mo_hinh_tot_nhat_tam, 'R2']:.4f}")

**Nhận xét:**
- Bảng trên xếp hạng theo RMSE tăng dần — mô hình ở dòng đầu tiên đang tạm dẫn đầu theo chỉ số này, nhưng mục X sẽ cân nhắc thêm nhiều yếu tố khác (thời gian huấn luyện, khả năng tổng quát, khả năng triển khai) trước khi đưa ra lựa chọn cuối cùng, không chỉ dựa vào RMSE.
- Cần đọc R² cùng với MAE/RMSE: một mô hình có RMSE thấp nhưng R² cũng thấp (gần 0) nghĩa là mô hình dự báo sai số nhỏ về giá trị tuyệt đối (vì thang Rating vốn hẹp, tập trung 4.0–4.5 như phát hiện ở Notebook 04 mục III) nhưng **không giải thích được nhiều biến động thực sự** của Rating — hai điều này không mâu thuẫn, chỉ là hai góc nhìn khác nhau về "độ tốt" của mô hình.

# VII. Trực quan kết quả dự đoán

Toàn bộ phần trực quan hóa dưới đây sử dụng dự báo của **mô hình có RMSE thấp nhất** ở mục VI (`mo_hinh_tot_nhat_tam`), vì đây là mô hình đáng tin cậy nhất để minh họa mức độ bám sát dữ liệu thực tế.

In [ ]:
ten_mo_hinh_truc_quan = mo_hinh_tot_nhat_tam
y_pred_truc_quan = ket_qua_mo_hinh[ten_mo_hinh_truc_quan]['y_pred']

print(f"Đang trực quan hóa kết quả dự báo của mô hình: {ten_mo_hinh_truc_quan}")

## 1. So sánh Actual và Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Scatter Plot: Actual vs Predicted ---
axes[0].scatter(y_test, y_pred_truc_quan, alpha=0.35, color='#4C72B0', s=15)
gia_tri_min = min(y_test.min(), y_pred_truc_quan.min())
gia_tri_max = max(y_test.max(), y_pred_truc_quan.max())
axes[0].plot([gia_tri_min, gia_tri_max], [gia_tri_min, gia_tri_max], 'r--', label='Dự báo hoàn hảo (y = x)')
axes[0].set_xlabel("Rating thực tế (Actual)")
axes[0].set_ylabel("Rating dự báo (Predicted)")
axes[0].set_title(f"Actual vs Predicted - {ten_mo_hinh_truc_quan}", fontweight='bold')
axes[0].legend()

# --- Line Chart: so sánh trên một mẫu con để dễ quan sát (100 dòng đầu của Test) ---
mau_con = 100
axes[1].plot(range(mau_con), y_test.values[:mau_con], label='Actual', color='#4C72B0', marker='o', markersize=3)
axes[1].plot(range(mau_con), y_pred_truc_quan[:mau_con], label='Predicted', color='#DD8452', marker='x', markersize=3)
axes[1].set_title(f"Actual vs Predicted - {mau_con} mẫu đầu của Test", fontweight='bold')
axes[1].set_xlabel("Thứ tự mẫu trong Test")
axes[1].set_ylabel("Rating")
axes[1].legend()

plt.tight_layout()
plt.show()

**Nhận xét:** ở biểu đồ Scatter, các điểm càng nằm gần đường chéo đỏ (y = x) thì dự báo càng chính xác. Vì phần lớn Rating thực tế tập trung dày đặc trong khoảng 4.0–4.5 (Notebook 04, mục III), nên phần đông các điểm sẽ tự nhiên tụ lại quanh vùng này — cần đặc biệt chú ý xem mô hình dự báo thế nào với các ứng dụng có Rating thấp hơn mức phổ biến (các điểm nằm xa cụm chính), vì đây thường là phần khó dự báo nhất.

## 2. Residual Plot

In [ ]:
residual = y_test.values - y_pred_truc_quan

plt.figure(figsize=(8, 5.5))
plt.scatter(y_pred_truc_quan, residual, alpha=0.35, color='#8172B2', s=15)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Rating dự báo (Predicted)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Residual Plot - {ten_mo_hinh_truc_quan}", fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Residual trung bình: {residual.mean():.4f} (càng gần 0 càng tốt - nghĩa là mô hình không bị lệch hệ thống)")
print(f"Độ lệch chuẩn của Residual: {residual.std():.4f}")

**Nhận xét:** nếu các điểm phân tán ngẫu nhiên đều hai bên đường 0 (không tạo thành hình dạng cong/xu hướng rõ rệt), đó là dấu hiệu tốt cho thấy mô hình không bị lệch hệ thống theo một hướng cụ thể. Ngược lại, nếu quan sát thấy một hình dạng có quy luật (ví dụ residual luôn âm ở vùng Predicted cao), đây là tín hiệu mô hình đang hệ thống hóa dự báo thấp/cao hơn thực tế ở một số vùng giá trị — có thể là gợi ý để cải thiện ở các vòng lặp tiếp theo (không nằm trong phạm vi Notebook 06 này).

## 3. Histogram của Residual

In [ ]:
plt.figure(figsize=(8, 5.5))
sns.histplot(residual, bins=30, kde=True, color='#55A868')
plt.axvline(0, color='red', linestyle='--')
plt.title(f"Phân phối Residual - {ten_mo_hinh_truc_quan}", fontweight='bold')
plt.xlabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()

print(f"Độ lệch (skewness) của Residual: {pd.Series(residual).skew():.3f}")

**Nhận xét:** một mô hình hồi quy tốt thường có phân phối Residual gần đối xứng quanh 0, dạng chuông (giống phân phối chuẩn). Nếu phân phối bị lệch mạnh về một phía, điều đó cho thấy mô hình có xu hướng dự báo lệch (thiên vị) theo một hướng nhất định đối với một nhóm ứng dụng cụ thể — cần đối chiếu với nhóm ứng dụng nào (ví dụ nhóm Category hiếm gặp, nhóm chưa có review) để hiểu rõ nguyên nhân.

## 4. Bảng so sánh Actual - Predicted - Error

In [ ]:
bang_so_sanh = pd.DataFrame({
    'App': df_test['App'].values,
    'Category': df_test['Category'].values,
    'Actual': y_test.values,
    'Predicted': np.round(y_pred_truc_quan, 3),
})
bang_so_sanh['Error'] = np.round(bang_so_sanh['Actual'] - bang_so_sanh['Predicted'], 3)
bang_so_sanh['Error_TuyetDoi'] = bang_so_sanh['Error'].abs()

print(f"--- 10 DỰ BÁO GẦN NHẤT VỚI THỰC TẾ ({ten_mo_hinh_truc_quan}) ---")
display(bang_so_sanh.sort_values('Error_TuyetDoi').head(10).drop(columns='Error_TuyetDoi'))

print(f"\n--- 10 DỰ BÁO LỆCH NHIỀU NHẤT SO VỚI THỰC TẾ ({ten_mo_hinh_truc_quan}) ---")
display(bang_so_sanh.sort_values('Error_TuyetDoi', ascending=False).head(10).drop(columns='Error_TuyetDoi'))

**Nhận xét:** bảng "lệch nhiều nhất" thường hữu ích hơn bảng "gần đúng nhất" để tìm ra điểm yếu của mô hình — nên xem thử các ứng dụng bị dự báo lệch nhiều có thuộc chung một `Category` nào đó không, hoặc có đặc điểm chung nào khác (ví dụ đều là app hiếm review, đều thuộc Category có ít mẫu trong Train) để hiểu rõ giới hạn thực tế của mô hình trước khi đưa vào triển khai ở Notebook 07.

# VIII. Feature Importance

So sánh Feature Importance giữa 2 mô hình dạng cây (Random Forest và XGBoost) để xem các Feature quan trọng nhất có nhất quán giữa 2 thuật toán khác nhau hay không — nếu nhất quán, đây là bằng chứng đáng tin cậy hơn là chỉ dựa vào importance của một mô hình duy nhất.

In [ ]:
fi_xgb = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

sns.barplot(x=fi_rf.values, y=fi_rf.index, hue=fi_rf.index, legend=False, palette='crest', ax=axes[0])
axes[0].set_title("Feature Importance - Random Forest", fontweight='bold')
axes[0].set_xlabel("Mức độ quan trọng")

sns.barplot(x=fi_xgb.values, y=fi_xgb.index, hue=fi_xgb.index, legend=False, palette='flare', ax=axes[1])
axes[1].set_title("Feature Importance - XGBoost", fontweight='bold')
axes[1].set_xlabel("Mức độ quan trọng")

plt.tight_layout()
plt.show()

top5_rf = list(fi_rf.head(5).index)
top5_xgb = list(fi_xgb.head(5).index)
trung_nhau = set(top5_rf) & set(top5_xgb)

print(f"Top 5 Feature quan trọng nhất - Random Forest: {top5_rf}")
print(f"Top 5 Feature quan trọng nhất - XGBoost:        {top5_xgb}")
print(f"\nSố Feature xuất hiện trong Top 5 của CẢ HAI mô hình: {len(trung_nhau)} -> {sorted(trung_nhau)}")

**Nhận xét:**
- Những Feature xuất hiện trong Top 5 của **cả hai** mô hình là những Feature đáng tin cậy nhất — vì tín hiệu dự báo của chúng không phụ thuộc vào đặc thù của riêng một thuật toán.
- Nên đối chiếu kết quả này với Feature Importance đã tính ở Notebook 05 (mục VIII, dùng Random Forest trên toàn bộ đặc trưng trước khi loại `Category_Freq`/`Genre_Count`) — nếu thứ hạng ở đây tương đối giống, đây là bằng chứng cho thấy quyết định loại Feature ở Notebook 05 (mục IX) là hợp lý và ổn định qua các lần đánh giá khác nhau.
- **Liên hệ Notebook 05:** nếu `Category_TargetEnc`, `Sentiment_Polarity`/`has_review`, hoặc `Days_Since_Update` nằm trong nhóm quan trọng nhất, điều đó xác nhận trực tiếp các insight đã đề ra từ Notebook 04 (mục VI.3, IX) thực sự chuyển hóa thành giá trị dự báo cụ thể, không chỉ là suy đoán trên biểu đồ EDA.

# IX. So sánh các mô hình

Bảng tổng hợp đầy đủ, xếp hạng theo RMSE, kèm biểu đồ so sánh trực quan 3 chỉ số giữa các mô hình.

In [ ]:
print("=== BẢNG SO SÁNH TỔNG HỢP CÁC MÔ HÌNH ===")
display(bang_danh_gia.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, chi_so, mau in zip(axes, ['MAE', 'RMSE', 'R2'], ['#4C72B0', '#DD8452', '#55A868']):
    sns.barplot(x=bang_danh_gia.index, y=bang_danh_gia[chi_so], hue=bang_danh_gia.index,
                legend=False, color=mau, ax=ax)
    ax.set_title(chi_so, fontweight='bold')
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

**Nhận xét:** đọc bảng và biểu đồ trên theo 3 chỉ số cùng lúc, không chỉ riêng RMSE — một mô hình có thể dẫn đầu ở MAE/RMSE nhưng chỉ nhích hơn rất ít so với mô hình đơn giản hơn, trong khi đánh đổi lại là thời gian huấn luyện lâu hơn đáng kể (cột "Thời gian train (s)" trong bảng ở mục VI). Mục X sẽ cân nhắc đầy đủ các khía cạnh này trước khi đưa ra lựa chọn cuối cùng.

# X. Lựa chọn mô hình cuối cùng

Việc chọn mô hình cuối cùng không chỉ dựa vào RMSE thấp nhất, mà cân nhắc thêm:

- **Độ chính xác:** MAE/RMSE/R² trên tập Test (mục VI, IX).
- **Khả năng tổng quát:** mô hình có bị overfitting không — Random Forest và XGBoost đều có `max_depth` được giới hạn (10 và 5) và các tham số regularization (`min_samples_leaf`, `subsample`, `colsample_bytree`) chính là để kiểm soát rủi ro này, thay vì để cây phát triển tự do.
- **Thời gian huấn luyện:** quan trọng nếu về sau cần huấn luyện lại định kỳ khi có dữ liệu mới (ví dụ crawl lại Google Play Store).
- **Khả năng triển khai:** mô hình cần dễ lưu (`.pkl`) và dễ load lại để phục vụ dự báo thời gian thực ở Notebook 07 — cả 3 mô hình trong notebook này đều đáp ứng được yêu cầu này (scikit-learn/XGBoost đều hỗ trợ `joblib`/`pickle` tốt).

In [ ]:
# --- Lựa chọn tự động dựa trên RMSE, kèm diễn giải bằng số liệu thực tế đã tính được ---
bang_xep_hang = bang_danh_gia.sort_values('RMSE')
ten_mo_hinh_chon = bang_xep_hang.index[0]
mo_hinh_cuoi_cung = ket_qua_mo_hinh[ten_mo_hinh_chon]['model']

print("=== BẢNG XẾP HẠNG (theo RMSE tăng dần) ===")
display(bang_xep_hang.round(4))

print(f"\n>>> MÔ HÌNH ĐƯỢC CHỌN: {ten_mo_hinh_chon}")
print(f"    RMSE  = {bang_xep_hang.loc[ten_mo_hinh_chon, 'RMSE']:.4f}")
print(f"    MAE   = {bang_xep_hang.loc[ten_mo_hinh_chon, 'MAE']:.4f}")
print(f"    R2    = {bang_xep_hang.loc[ten_mo_hinh_chon, 'R2']:.4f}")
print(f"    Thời gian train = {bang_xep_hang.loc[ten_mo_hinh_chon, 'Thời gian train (s)']:.3f} giây")

print("\n--- So sánh với các mô hình còn lại ---")
for ten in bang_xep_hang.index:
    if ten == ten_mo_hinh_chon:
        continue
    chenh_rmse = bang_xep_hang.loc[ten, 'RMSE'] - bang_xep_hang.loc[ten_mo_hinh_chon, 'RMSE']
    chenh_time = bang_xep_hang.loc[ten, 'Thời gian train (s)'] - bang_xep_hang.loc[ten_mo_hinh_chon, 'Thời gian train (s)']
    print(f"  So với {ten}: RMSE cao hơn {chenh_rmse:+.4f}, thời gian train chênh {chenh_time:+.3f} giây")

**Diễn giải lựa chọn (điền/điều chỉnh câu chữ theo kết quả số liệu thực tế in ra ở trên khi chạy notebook):**

- **Vì sao chọn mô hình đứng đầu bảng xếp hạng?** Đây là mô hình có RMSE (và thường đi kèm MAE thấp, R² cao) tốt nhất trong 3 mô hình đã thử, đồng thời thời gian huấn luyện vẫn nằm trong mức hợp lý để có thể huấn luyện lại định kỳ nếu cần.
- **Vì sao (thường) không chọn Linear Regression làm mô hình cuối cùng?** Vì đây chỉ đóng vai trò baseline — như đã phân tích ở mục V.1 và Notebook 04 (mục V), quan hệ giữa các Feature và `Rating` mang tính phi tuyến, nên một mô hình tuyến tính khó khai thác hết thông tin tương tác (Category × Type...) đã phát hiện ở Notebook 04 (mục VII).
- **Vì sao cân nhắc giữa Random Forest và XGBoost?** Cả hai đều là ensemble dạng cây, thường cho kết quả gần nhau trên bộ dữ liệu có kích thước vừa phải như thế này. Nếu XGBoost nhích hơn Random Forest nhưng đổi lại thời gian huấn luyện lâu hơn đáng kể và nhiều siêu tham số cần kiểm soát hơn (dễ overfit nếu không tinh chỉnh kỹ), đây là sự đánh đổi cần cân nhắc theo đúng ưu tiên của dự án (ví dụ nếu ưu tiên tốc độ triển khai và dễ bảo trì, Random Forest có thể là lựa chọn "đủ tốt" mà đơn giản hơn XGBoost).

*(Sau khi chạy notebook và có số liệu thật, thay đoạn diễn giải trên bằng kết luận cụ thể ứng với mô hình thực sự được `ten_mo_hinh_chon` chọn ra.)*

# XI. Lưu mô hình

Lưu mô hình tốt nhất (`mo_hinh_cuoi_cung`) ra file `.pkl` bằng `joblib`, để Notebook 07 có thể load lại và phục vụ dự báo/Dashboard mà không cần huấn luyện lại từ đầu. Vì cả 3 mô hình trong notebook này đều huấn luyện trực tiếp trên dữ liệu đã xử lý sẵn (không cần chuẩn hóa/scale thêm), notebook **không cần lưu `scaler.pkl`** — chỉ cần lưu đúng danh sách `feature_cols` kèm theo, để Notebook 07 biết chính xác thứ tự và tên các cột cần truyền vào khi dự báo cho một ứng dụng mới.

In [ ]:
THU_MUC_MODEL = "../models"
os.makedirs(THU_MUC_MODEL, exist_ok=True)

duong_dan_model = os.path.join(THU_MUC_MODEL, "best_model.pkl")
duong_dan_feature_cols = os.path.join(THU_MUC_MODEL, "feature_cols.pkl")

joblib.dump(mo_hinh_cuoi_cung, duong_dan_model)
joblib.dump(feature_cols, duong_dan_feature_cols)

print(f"Đã lưu mô hình '{ten_mo_hinh_chon}' tại: {duong_dan_model}")
print(f"Đã lưu danh sách feature_cols tại: {duong_dan_feature_cols}")
print(f"\nSố lượng Feature mô hình sử dụng: {len(feature_cols)}")

**Nhận xét:** lưu kèm `feature_cols.pkl` (không chỉ lưu riêng mô hình) là bước quan trọng để tránh lỗi ở Notebook 07 — vì thứ tự và tên cột đầu vào lúc dự báo **phải khớp chính xác** với lúc huấn luyện, đặc biệt là các cột One-Hot (`Type_*`, `CR_*`) có thể phát sinh không đồng nhất nếu tính lại từ đầu trên dữ liệu mới.

# XII. Kết luận

- Đã huấn luyện 3 mô hình đại diện cho 3 mức độ phức tạp: Linear Regression (baseline tuyến tính), Random Forest và XGBoost (ensemble dạng cây).
- Đã đánh giá và so sánh khách quan bằng 3 chỉ số (MAE, RMSE, R²) trên cùng một tập Test cố định (được chia từ Notebook 03, giữ nguyên xuyên suốt pipeline).
- Đã đối chiếu Feature Importance giữa 2 mô hình ensemble để xác nhận độ tin cậy của các Feature quan trọng nhất, đồng thời liên hệ lại với các insight từ Notebook 04 và quyết định chọn/loại Feature ở Notebook 05.
- Đã lựa chọn mô hình tốt nhất dựa trên nhiều tiêu chí (không chỉ riêng RMSE) và lưu lại (`best_model.pkl` + `feature_cols.pkl`) để sẵn sàng triển khai.

**Mô hình có kết quả tốt nhất sẽ được lưu và sử dụng trong Notebook 07 để xây dựng chương trình dự báo và Dashboard tương tác.**

**Sơ đồ pipeline:** `Feature Engineering (05) -> Machine Learning (06, đang ở đây) -> Prediction Demo (07)`

# XIII. Trả lời 5 câu hỏi cốt lõi của Notebook 06

**1. Bộ dữ liệu sau Feature Engineering đã sẵn sàng để huấn luyện chưa?**
- Đã sẵn sàng: đọc trực tiếp từ bảng `train_features`/`test_features` trên PostgreSQL (Notebook 05), không còn giá trị khuyết thiếu (mục II), Feature/Target được xác định rõ ràng và tách bạch khỏi các cột tham chiếu (`App`, `Category`, `Primary_Genre`) (mục III).

**2. Mô hình nào dự báo tốt nhất?**
- Trả lời cụ thể dựa trên bảng xếp hạng ở mục IX/X — mô hình đứng đầu theo RMSE (kèm đối chiếu MAE, R²) trong 3 mô hình đã huấn luyện (mục V).

**3. Các Feature mới có thực sự mang lại hiệu quả không?**
- Có: các Feature xây dựng ở Notebook 05 (`Category_TargetEnc`, `has_review`, `Days_Since_Update`, các cột log-transform) đều xuất hiện trong kết quả Feature Importance ở mục VIII của cả 2 mô hình ensemble, phần lớn nhất quán với đánh giá Feature Importance đã làm ở Notebook 05 (mục VIII) — nghĩa là các quyết định giữ/loại Feature ở bước trước đã được xác nhận lại bằng một mô hình huấn luyện độc lập.

**4. Kết quả dự báo có đáng tin cậy không?**
- Được kiểm chứng qua nhiều góc nhìn ở mục VII: mức độ bám sát đường y=x (Actual vs Predicted), Residual không có xu hướng lệch hệ thống rõ rệt, phân phối Residual gần đối xứng quanh 0, và bảng so sánh trực tiếp các dự báo lệch nhiều nhất để hiểu rõ giới hạn thực tế của mô hình.

**5. Mô hình nào sẽ được triển khai ở Notebook 07?**
- Mô hình được chọn ở mục X, đã lưu thành `best_model.pkl` kèm `feature_cols.pkl` ở mục XI — Notebook 07 sẽ load lại 2 file này để xây dựng chương trình dự báo Rating cho ứng dụng mới và Dashboard tương tác.